# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells: Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² dataset on protein abundance and modification analysis from human cells, using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset via `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata and print basic info
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset name:", metadata.name)
print("Description:", metadata.description)


## 2. Data Overview
Enumerate available record sets, their fields, and associated `@id` values.

In [ ]:
# List all record sets and their fields, using their @id
print("Available record sets (by @id):")
record_sets = list(metadata.record_sets)
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}, name: {rs.name}")
    if hasattr(rs, "fields"):
        for f in rs.fields:
            print(f"    - Field @id: {f.id}, name: {f.name}, dataType: {f.data_type}")
    else:
        print("    (No fields information)")

## 3. Data Extraction
Load data from a chosen record set into a DataFrame for analysis. Here we use the primary protein data table, using `@id` referencing as per Croissant best practices.

In [ ]:
# Choose record set(s) to load: find the relevant @id from the overview above
# For this dataset, the main protein table is typically named with 'ProteinTable' in id or name

# Extract all RecordSet @ids for reference
record_set_ids = [rs.id for rs in record_sets]
print("RecordSet IDs:", record_set_ids)

# (You may need to adjust this depending on the printed output)
# Example: we use the first record set as the main table for demonstration
main_record_set_id = record_set_ids[0] if record_set_ids else None

dataframes = {}

for rec_id in record_set_ids:
    recs = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(recs)
    dataframes[rec_id] = df
    print(f"Loaded DataFrame for RecordSet {rec_id}: shape = {df.shape}")

if main_record_set_id:
    print("Columns in main record set:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We'll now select a numeric field (e.g., 'mw' for molecular weight, or a peptide count/abundance column) by its `@id`, filter records, normalize, and group by some categorical variable.

In [ ]:
# List all numeric fields in the chosen record set
df = dataframes[main_record_set_id]
print("Possible numeric fields:")
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        print(f"- {col}")

# We'll select one (e.g. molecular weight), else pick first numeric
preferred_numeric = None
for candidate in [
    'mw',  # common for molecular weight
    'num_peptides',  # peptide count
    'abundance',  # some normalized or total abundance
]:
    if candidate in df.columns:
        preferred_numeric = candidate
        break

if not preferred_numeric:
    # fallback: pick the first numeric column
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    preferred_numeric = numeric_cols[0] if numeric_cols else None

numeric_field_id = preferred_numeric
print(f"Selected numeric field (by name/@id): {numeric_field_id}")

# Filtering on the numeric field, e.g. MW > threshold
threshold = df[numeric_field_id].mean() if numeric_field_id else 10
filtered_df = df[df[numeric_field_id] > threshold] if numeric_field_id else df.copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (n={len(filtered_df)}):")
display(filtered_df.head())

# Normalizing the numeric field
if numeric_field_id:
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a textual/categorical field if available
group_field = None
# Look for 'description', 'protein_class', or choose any non-numeric suitable field
for candidate in ['description', 'protein_class', 'accession']:
    if candidate in df.columns and (not pd.api.types.is_numeric_dtype(df[candidate])):
        group_field = candidate
        break
if not group_field:
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            group_field = col
            break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
    print(f"Grouped data by '{group_field}', mean {numeric_field_id}:")
    display(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Show distributions and relationships between fields using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if numeric_field_id:
    # Histogram of numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=40)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If a group_field exists and has reasonable cardinality, boxplot
    if group_field and df[group_field].nunique() < 40:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=df[group_field], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored extracellular vesicle proteomic data using `mlcroissant`. You can adapt this notebook to examine specific protein classes, relationships between sample attributes, or to prepare data for machine learning workflows using the Croissant-provided structure.

**Key findings:**
- The dataset includes structured records for protein abundance and description, with several numeric and string fields.
- Filtering and normalization by fields such as molecular weight or peptide count are straightforward.
- Categorical grouping (e.g., by protein annotations) is supported via @ids and column names exposed in the schema.

> For more advanced analyses, extend this notebook to add machine learning tasks, outlier detection, or deeper biological categorization as needed!